# 📅 Notebook 3: Sliding-Window Counter

A common production design — used by Cloudflare, GitHub, Stripe — is a **sliding-window log**: keep timestamps of recent requests; allow a new request iff fewer than `N` fall within the last `W` seconds.

More accurate than a **fixed-window** counter (which has a 2× burst at window boundaries) but slightly more memory.


## 🛠️ Setup

```bash
cd 04-patterns/rate-limiting-and-throttling
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟥 Naive fixed-window counter — boundary burst

In [ ]:
import time
from collections import deque

class FixedWindow:
    def __init__(self, limit, window):
        self.limit = limit; self.window = window
        self.start = time.monotonic(); self.count = 0
    def allow(self):
        now = time.monotonic()
        if now - self.start >= self.window:
            self.start = now; self.count = 0
        if self.count < self.limit:
            self.count += 1; return True
        return False

fw = FixedWindow(limit=5, window=1.0)
# At t=0.99 a burst of 5 fits; at t=1.01 a fresh window allows 5 more = 10 in ~20ms!
print('attempts inside fake-late-window:', [int(fw.allow()) for _ in range(5)])
fw.start -= 1.0  # pretend the window just rolled
print('attempts inside fake-fresh-window:', [int(fw.allow()) for _ in range(5)])


## 🟩 Sliding-window log — correct everywhere

In [ ]:
class SlidingWindow:
    def __init__(self, limit, window):
        self.limit = limit; self.window = window
        self.events = deque()
    def allow(self):
        now = time.monotonic()
        cutoff = now - self.window
        while self.events and self.events[0] < cutoff:
            self.events.popleft()
        if len(self.events) < self.limit:
            self.events.append(now); return True
        return False

sw = SlidingWindow(limit=5, window=1.0)
results = []
for _ in range(8):
    results.append(int(sw.allow())); time.sleep(0.05)
print('first 8 attempts:', results)
time.sleep(1.0)
print('after 1s:', [int(sw.allow()) for _ in range(5)])


## 📊 Algorithm comparison

| Algorithm | Memory | Burst behavior | Boundary safe? |
|---|---|---|---|
| Fixed window counter | O(1) | up to 2× limit at boundary | ❌ |
| Sliding window log | O(N) timestamps | smooth, exact | ✅ |
| Token bucket | O(1) | bounded burst | ✅ |
| Leaky bucket | O(1) | no burst (paced) | ✅ |

For a distributed limiter, replace the in-memory state with **Redis** counters (`INCR` + TTL for fixed window, sorted sets for sliding window).